In [3]:
# %% [markdown]
# # 03 - News collection: GDELT sentiment + SEC EDGAR filings
# Collects a market-wide daily tone index (GDELT) and firm-specific filing
# events (SEC EDGAR) for 5 tickers, keyed on `date_only` to match
# all_tickers_2020_2025.csv. Runs fully on HTTP requests - no GCP/BigQuery
# credentials needed, so it works as-is on Kaggle with internet access
# enabled in notebook settings (Settings -> Internet -> On).

In [1]:
import os
import time
import random
import requests
import pandas as pd

url = "https://api.gdeltproject.org/api/v2/doc/doc"

ticker = "NVDA"
company = '"NVIDIA"'

start_year = 2020
end_year = 2025

output_file = f"/kaggle/working/gdelt_{ticker.lower()}_2020_2025.csv"

expected_days = {
    2020: 366,
    2021: 365,
    2022: 365,
    2023: 365,
    2024: 366,
    2025: 365
}

In [2]:
if os.path.exists(output_file):

    existing_df = pd.read_csv(output_file)
    existing_df["date_only"] = pd.to_datetime(existing_df["date_only"])

    existing_df = (existing_df.drop_duplicates(subset=["ticker", "date_only"]).sort_values("date_only").reset_index(drop=True))

    print("Existing file found.")
    print("Existing rows:", len(existing_df))

else:
    existing_df = pd.DataFrame(columns=["date_only", "tone", "ticker"])
    print("No existing output file found.")

No existing output file found.


In [3]:
minimum_days_per_year = 340
year_counts = {}
if not existing_df.empty:
    year_counts = (existing_df.groupby(existing_df["date_only"].dt.year).size().to_dict())

years_to_fetch = []
for year in range(start_year, end_year + 1):
    collected_days = year_counts.get(year, 0)
    if collected_days < minimum_days_per_year:
        years_to_fetch.append(year)
        print(f"{year}: incomplete "f"({collected_days} rows) → will retrieve")

    else:
        print(f"{year}: already available "f"({collected_days} rows) → skipped")
print("\nYears to retrieve:", years_to_fetch)

2020: incomplete (0 rows) → will retrieve
2021: incomplete (0 rows) → will retrieve
2022: incomplete (0 rows) → will retrieve
2023: incomplete (0 rows) → will retrieve
2024: incomplete (0 rows) → will retrieve
2025: incomplete (0 rows) → will retrieve

Years to retrieve: [2020, 2021, 2022, 2023, 2024, 2025]


In [5]:
def fetch_gdelt_year(
    year,
    ticker,
    company,
    max_attempts=12,
    base_wait=30
):

    params = {
        "query": company,
        "mode": "timelinetone",
        "format": "json",
        "startdatetime": f"{year}0101000000",
        "enddatetime": f"{year}1231235959"
    }

    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=120,
                headers={
                    "User-Agent":
                    "Mozilla/5.0 academic-research-project/1.0"})

            print(
                f"{year} | attempt {attempt}/{max_attempts} "
                f"| status {response.status_code}")

            if response.status_code == 200:
                data = response.json()
                rows = []

                for series in data.get("timeline", []):
                    for point in series.get("data", []):
                        rows.append({
                            "date_only": point["date"][:8],
                            "tone": point["value"],
                            "ticker": ticker})

                if rows:
                    year_df = pd.DataFrame(rows)
                    year_df["date_only"] = pd.to_datetime(
                        year_df["date_only"],
                        format="%Y%m%d")

                    year_df = (year_df.drop_duplicates(subset=["ticker", "date_only"]).sort_values("date_only").reset_index(drop=True))

                    print(f"{year}: successfully collected " f"{len(year_df)} rows")
                    return year_df
                print(f"{year}: status 200 but no timeline data returned")

            elif response.status_code == 429:
                wait_time = (base_wait * attempt + random.randint(10, 30))
                print(f"Rate limited. Waiting "f"{wait_time} seconds...")
                time.sleep(wait_time)

            else:
                wait_time = (base_wait + random.randint(10, 30))
                print(f"Request failed. Waiting "f"{wait_time} seconds...")
                time.sleep(wait_time)

        except requests.exceptions.RequestException as error:
            wait_time = (base_wait * attempt + random.randint(10, 30))

            print("Request error:", error)
            print(f"Waiting {wait_time} seconds...")
            time.sleep(wait_time)

        except ValueError as error:
            wait_time = (base_wait + random.randint(10, 30))
            print("JSON parsing error:", error)
            print(f"Waiting {wait_time} seconds...")
            time.sleep(wait_time)

    print(f"{year}: failed after "f"{max_attempts} attempts")
    return pd.DataFrame()

In [7]:
combined_df = existing_df.copy()

for year in years_to_fetch:

    print(f"\n{'=' * 50}")
    print(f"Retrieving {ticker} for {year}")
    print(f"{'=' * 50}")

    year_df = fetch_gdelt_year(
        year=year,
        ticker=ticker,
        company=company,
        max_attempts=12,
        base_wait=30
    )

    if not year_df.empty:
        combined_df = pd.concat(
            [combined_df, year_df],
            ignore_index=True
        )

        combined_df["date_only"] = pd.to_datetime(
            combined_df["date_only"]
        )

        combined_df = (
            combined_df
            .drop_duplicates(
                subset=["ticker", "date_only"],
                keep="last"
            )
            .sort_values(["ticker", "date_only"])
            .reset_index(drop=True)
        )

        combined_df.to_csv(
            output_file,
            index=False
        )

        print(
            f"Checkpoint saved after {year}: "
            f"{output_file}"
        )

    else:

        print(
            f"{year} was not collected. "
            f"Continuing to the next year."
        )

    # Rest before the next yearly request
    time.sleep(120)


Retrieving NVDA for 2020
2020 | attempt 1/12 | status 429
Rate limited. Waiting 58 seconds...
2020 | attempt 2/12 | status 200
2020: successfully collected 365 rows
Checkpoint saved after 2020: /kaggle/working/gdelt_nvda_2020_2025.csv


/tmp/ipykernel_58/224092558.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(



Retrieving NVDA for 2021
2021 | attempt 1/12 | status 429
Rate limited. Waiting 47 seconds...
2021 | attempt 2/12 | status 429
Rate limited. Waiting 80 seconds...
2021 | attempt 3/12 | status 429
Rate limited. Waiting 100 seconds...
2021 | attempt 4/12 | status 429
Rate limited. Waiting 150 seconds...
2021 | attempt 5/12 | status 200
2021: successfully collected 365 rows
Checkpoint saved after 2021: /kaggle/working/gdelt_nvda_2020_2025.csv

Retrieving NVDA for 2022
2022 | attempt 1/12 | status 429
Rate limited. Waiting 57 seconds...
2022 | attempt 2/12 | status 429
Rate limited. Waiting 72 seconds...
2022 | attempt 3/12 | status 429
Rate limited. Waiting 115 seconds...
2022 | attempt 4/12 | status 429
Rate limited. Waiting 130 seconds...
2022 | attempt 5/12 | status 429
Rate limited. Waiting 173 seconds...
2022 | attempt 6/12 | status 429
Rate limited. Waiting 201 seconds...
2022 | attempt 7/12 | status 200
2022: successfully collected 365 rows
Checkpoint saved after 2022: /kaggle/wor

In [8]:
if os.path.exists(output_file):

    final_df = pd.read_csv(output_file)

    final_df["date_only"] = pd.to_datetime(
        final_df["date_only"]
    )

    coverage = (
        final_df
        .groupby(final_df["date_only"].dt.year)
        .size()
        .rename("rows")
        .reset_index()
        .rename(columns={"date_only": "year"})
    )

    display(coverage)

    print("Total rows:", len(final_df))
    print("First date:", final_df["date_only"].min())
    print("Last date:", final_df["date_only"].max())

    print(
        "Duplicate ticker-dates:",
        final_df.duplicated(
            subset=["ticker", "date_only"]
        ).sum()
    )

else:

    print("No output file was produced.")

,year,rows
0,2020,365
1,2021,365
2,2022,365
3,2023,364
4,2024,366
5,2025,348


Total rows: 2173
First date: 2020-01-01 00:00:00
Last date: 2025-12-31 00:00:00
Duplicate ticker-dates: 0


Inspect and found missing date and data. 

| Year | Expected calendar days | Collected | Missing |
|------|------------------------|-----------|---------|
| 2020 | 366 | 365 | 1 |
| 2021 | 365 | 365 | 0 |
| 2022 | 365 | 365 | 0 |
| 2023 | 365 | 364 | 1 |
| 2024 | 366 | 366 | 0 |
| 2025 | 365 | 348 | 17 |

In [9]:
check_df = pd.read_csv("gdelt_nvda_2020_2025.csv")

check_df["date_only"] = pd.to_datetime(
    check_df["date_only"]
)

for year in range(2020, 2026):

    expected_dates = pd.date_range(
        start=f"{year}-01-01",
        end=f"{year}-12-31",
        freq="D"
    )

    collected_dates = check_df.loc[
        check_df["date_only"].dt.year == year,
        "date_only"
    ]

    missing_dates = expected_dates.difference(
        collected_dates
    )

    print(f"\n{year}: {len(missing_dates)} missing dates")

    if len(missing_dates) > 0:
        print(missing_dates.tolist())


2020: 1 missing dates
[Timestamp('2020-10-20 00:00:00')]

2021: 0 missing dates

2022: 0 missing dates

2023: 1 missing dates
[Timestamp('2023-03-23 00:00:00')]

2024: 0 missing dates

2025: 17 missing dates
[Timestamp('2025-06-15 00:00:00'), Timestamp('2025-06-16 00:00:00'), Timestamp('2025-06-17 00:00:00'), Timestamp('2025-06-18 00:00:00'), Timestamp('2025-06-19 00:00:00'), Timestamp('2025-06-20 00:00:00'), Timestamp('2025-06-21 00:00:00'), Timestamp('2025-06-22 00:00:00'), Timestamp('2025-06-23 00:00:00'), Timestamp('2025-06-24 00:00:00'), Timestamp('2025-06-25 00:00:00'), Timestamp('2025-06-26 00:00:00'), Timestamp('2025-06-27 00:00:00'), Timestamp('2025-06-28 00:00:00'), Timestamp('2025-06-29 00:00:00'), Timestamp('2025-06-30 00:00:00'), Timestamp('2025-07-01 00:00:00')]


In [10]:
print(
    check_df.groupby(
        check_df["date_only"].dt.year
    )["date_only"].agg(
        ["min", "max", "count"]
    )
)

                 min        max  count
date_only                             
2020      2020-01-01 2020-12-31    365
2021      2021-01-01 2021-12-31    365
2022      2022-01-01 2022-12-31    365
2023      2023-01-01 2023-12-31    364
2024      2024-01-01 2024-12-31    366
2025      2025-01-01 2025-12-31    348
